# 🚀 Colab Ollama MCP — Option 2: Performance Multi-Model (Tesla T4 16GB / A100)

> **Lựa chọn chính (Daily Performance):** Đảm bảo chất lượng output ở mức **Top-Tier / Middle+** cho các bài toán phức tạp, refactor nhiều file và suy luận logic sâu. Tiết kiệm CU (~1.53 CU/h trên Tesla T4) giúp bạn code **3–4 tiếng/ngày suốt cả tháng** (chỉ tốn ~160 CU / 192 CU).

---

## 🧩 Bộ Đôi Mô Hình Cao Cấp (Option 2):

| Scope | Model Tag | Hãng | VRAM | Điểm mạnh cốt lõi |
| :--- | :--- | :--- | :---: | :--- |
| **1. Planning** | **`deepseek-r1:14b`** | **DeepSeek** | ~9.0 GB | Chuỗi tư duy `<think>` sâu sắc nhất phân khúc 14B, bóc tách kiến trúc, tìm bug logic & edge-cases. |
| **2. Coding, Workflow & Diagram** | **`qwen2.5-coder:14b`** ⭐ | **Alibaba** | ~9.5 GB | Vua coding mid-size (HumanEval 86.4%), bắt lỗi chặt chẽ, tối ưu thuật toán & vẽ sơ đồ Mermaid.js chuẩn 100%. |

---

💡 **Khi cần siêu tiết kiệm / CU sắp hết (Option 1):** Mở notebook [`colab_ollama_t4_saver.ipynb`](colab_ollama_t4_saver.ipynb)!
🚀 **Khi cần sức mạnh tột đỉnh 32B trên A100 (Combo 2):** Mở notebook [`colab_ollama_a100_max.ipynb`](colab_ollama_a100_max.ipynb)!

### Step 1: Verify NVIDIA GPU & VRAM Allocation
Confirm that your Google Colab runtime has an attached NVIDIA GPU (Tesla T4 16GB or A100 40GB VRAM).

In [ ]:
# Step 1: Verify allocated NVIDIA GPU hardware and available VRAM
!nvidia-smi

### Step 2: Configure Model Storage (Google Drive or Fast Colab SSD)
> 💡 **Note:** When running inside the **IDE Colab Kernel**, Google Drive browser popups are skipped automatically and Ollama uses Colab's ultra-fast NVMe SSD (~500 MB/s download speed).

In [ ]:
import os

# Set to True if running in Safari/Chrome and you want to cache weights in Google Drive
# Automatically falls back to fast local Colab SSD when running inside the IDE Kernel
USE_GDRIVE = True

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Persistent Google Drive cache enabled at: {models_dir}")
    except Exception as e:
        print(f"ℹ️ Running inside IDE Kernel (Drive mount skipped: {e}).")
        print("⚡ Using fast Colab local SSD storage (/root/.ollama/models).")
else:
    print("⚡ Using fast Colab local SSD storage (/root/.ollama/models).")

### Step 3: Install Ollama & Start Background Daemon
Download the official Linux Ollama binary and launch `ollama serve` as a background daemon listening on `0.0.0.0:11434`. Automatically clears installation logs afterward to keep memory usage low.

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Install zstd compression utility and the official Ollama Linux binary
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Configure environment variables for optimal GPU inference
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"  # Keep model weights pinned in GPU VRAM for 24 hours

# 3. Launch 'ollama serve' asynchronously as a background daemon
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Clear heavy installation logs to prevent DOM memory bloat
clear_output(wait=True)

# 5. Poll local health endpoint until the Ollama daemon responds
time.sleep(3)
try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Ollama Daemon is online:", resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Failed to reach Ollama daemon on localhost:11434:", e)

### Step 4: Pull Coding LLM (`qwen2.5-coder:14b` ⭐) into GPU VRAM
Recommended models for **Google Colab Pro**:
- **`qwen2.5-coder:14b` ⭐ (Recommended):** ~9.5 GB VRAM — Fits comfortably on a Tesla T4 (16GB VRAM), consumes minimal Compute Units, and outperforms Gemini Flash in coding quality.
- **`qwen2.5-coder:32b`:** ~20 GB VRAM — Flagship open-weight coder (requires selecting **A100 40GB GPU** in Colab Pro runtime settings).
- **`qwen2.5-coder:7b`:** ~4.5 GB VRAM — Ultra-fast lightweight fallback.

In [ ]:
from IPython.display import clear_output

# Bộ đôi mô hình Option 2 (Performance Multi-Model)
PERFORMANCE_MODELS = [
    ("qwen2.5-coder:14b", "💻 Coding, Workflow & Diagram: Alibaba Flagship Mid-size (HumanEval 86.4%)"),
    ("deepseek-r1:14b",   "🧠 Planning: DeepSeek Deep CoT Reasoning (<think> mode)"),
]

print("⏳ Đang tải bộ mô hình Option 2 (Performance Multi-Model)...")
for tag, role in PERFORMANCE_MODELS:
    print(f"\n📦 Đang tải {tag} ({role})...")
    !ollama pull {tag}

clear_output(wait=True)
print("✅ Hoàn tất tải toàn bộ mô hình! Danh sách hiện có trong máy:")
!ollama list


### Step 5: Open Cloudflare Quick Tunnel & Connect to Antigravity (2 Ways)
Start `cloudflared` to establish a reverse HTTPS tunnel from Colab's private NAT (`localhost:11434`) to a public `*.trycloudflare.com` URL.

In [ ]:
import re
import subprocess
import time
from IPython.display import clear_output

# 1. Download and install the Cloudflare Tunnel binary (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Spawn cloudflared tunnel forwarding public HTTPS traffic to localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Parse stdout stream to extract the generated '*.trycloudflare.com' public URL
tunnel_url = None
print("⏳ Establishing Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

# Clear verbose logs to keep DOM memory minimal
clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 COLAB OLLAMA GPU TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 WAY 1 (Drop Link into Running Agy CLI / Chat):\n")
    print(f"   Just paste this URL directly into your running Agy chat:  {tunnel_url}")
    print("   (Agy will automatically update mcp_config.json, check health, and switch to Colab GPU!)\n")
    print("⚡ WAY 2 (Run Command in Mac Terminal):\n")
    print(f"   python3 /Users/marchhygge/MarchHygge/project/colab-ollama-mcp/colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Failed to obtain Cloudflare Tunnel URL. Please re-run this cell.")

### Step 6: Session Keep-Alive (Optional — For Browser / Long Sessions)
Plays a silent 500ms audio loop if running in a browser, keeping the Colab session active while you code.

In [ ]:
from IPython.display import HTML, display

# Render an HTML5 looping silent audio player to prevent browser tab freezing / auto-reload (F5)
display(HTML('''
<div style="padding: 14px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px; font-family: sans-serif;">
  <b style="color: #2e7d32;">🟢 Keep-Alive Active (Silent Audio Loop)</b>
  <p style="margin: 4px 0 8px 0; color: #1b5e20; font-size: 13px;">
     Keeps the Colab session active while you code in Antigravity IDE.
  </p>
  <audio loop autoplay controls style="height: 30px;">
    <source src="https://raw.githubusercontent.com/anars/blank-audio/master/500-milliseconds-of-silence.mp3" type="audio/mp3">
  </audio>
</div>
'''))

### Step 7: Server Resource Monitor (GPU VRAM, System RAM, Disk & Active Model)
When running Colab inside **Antigravity IDE**, run this cell anytime to inspect real-time **GPU VRAM**, **GPU Temperature/Load**, **System RAM**, **SSD Disk**, and **Ollama Models Pinned in VRAM**.
- Set `AUTO_REFRESH = False` (default) for an instant one-time snapshot.
- Set `AUTO_REFRESH = True` to turn this cell into a live dashboard refreshing every 5 seconds (uses `clear_output(wait=True)` so it adds **0 MB** of DOM memory).

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# Set to True if you want a live dashboard that refreshes every 5 seconds
AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 22) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_colab_telemetry() -> None:
    print("=" * 74)
    print(f"📊 COLAB GPU SERVER TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 74)

    # 1. GPU VRAM, Utilization & Temperature via nvidia-smi
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : Unable to query nvidia-smi ({e})")

    # 2. Colab System RAM via /proc/meminfo
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ System RAM   : Unable to read /proc/meminfo ({e})")

    # 3. Colab SSD Disk Usage
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Ollama Model Pinned in GPU VRAM (/api/ps)
    print("-" * 74)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"🔥 Active LLM   : {m_name} — Pinned in VRAM: {vram_gb:.2f} GB ({gpu_offload:.0f}% GPU)")
            else:
                print("💤 Active LLM   : Idle (0 GB in VRAM — will load into GPU on first prompt)")
    except Exception as e:
        print(f"🔴 Ollama Status: Offline or unreachable ({e})")
    print("=" * 74)

try:
    while True:
        clear_output(wait=True)
        print_colab_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Auto-refreshing every {REFRESH_SECONDS}s... (Click Stop ⏹️ on cell to pause)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Live telemetry stopped.")